In [1]:
import torch
import numpy as np

import os
import sys

sys.path.append(os.path.abspath(".."))

from models.vae_adain import Model as VAE
from default_config import cfg

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Using /home/ncaytuir/.cache/torch_extensions/py38_cu113 as PyTorch extensions root...
Detected CUDA files, patching ldflags
Emitting ninja build file /home/ncaytuir/.cache/torch_extensions/py38_cu113/emd_ext/build.ninja...
Building extension module emd_ext...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)
ninja: no work to do.
Loading extension module emd_ext...
load emd_ext time: 0.139s
utils/utils.py: USE_COMET=1, USE_WB=0


In [2]:
# Path to ckpt
ckpt_path = "/home/ncaytuir/data-local/LION_necs/lion_ckpt/unconditional/airplane/checkpoints/vae_only.pt"
cfg_path = "/home/ncaytuir/data-local/LION_necs/lion_ckpt/unconditional/airplane/cfg.yml"

shape1 = "/home/ncaytuir/data/Datasets/ShapeNetCore.v2.PC15k/02691156/val/1a54a2319e87bd4071d03b466c72ce41.npy"
shape2 = "/home/ncaytuir/data/Datasets/ShapeNetCore.v2.PC15k/02691156/train/1a04e3eab45ca15dd86060f189eb133.npy"
ouput_path = "/home/ncaytuir/data-local/LION_necs/output/latent_test04.npy"

cfg.merge_from_file(cfg_path)
args = cfg

# Load ckpt
ckpt = torch.load(ckpt_path, map_location=device)

# Instanciate model
vae = VAE(args).to(device)
vae.load_state_dict(ckpt["model"])
vae.eval()

#print("args", args)

2026-03-19 14:59:19.929 | INFO     | utils.model_helper:import_model:114 - import: models.shapelatent_modules.PointNetPlusEncoder


Using /home/ncaytuir/.cache/torch_extensions/py38_cu113 as PyTorch extensions root...
Detected CUDA files, patching ldflags
Emitting ninja build file /home/ncaytuir/.cache/torch_extensions/py38_cu113/_pvcnn_backend/build.ninja...
Building extension module _pvcnn_backend...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


2026-03-19 14:59:21.209 | INFO     | models.shapelatent_modules:__init__:29 - [Encoder] zdim=128, out_sigma=True; force_att: 0
2026-03-19 14:59:21.210 | INFO     | utils.model_helper:import_model:114 - import: models.latent_points_ada.PointTransPVC
2026-03-19 14:59:21.213 | INFO     | models.latent_points_ada:__init__:38 - [Build Unet] extra_feature_channels=0, input_dim=3
2026-03-19 14:59:21.321 | INFO     | utils.model_helper:import_model:114 - import: models.latent_points_ada.LatentPointDecPVC
2026-03-19 14:59:21.322 | INFO     | models.latent_points_ada:__init__:241 - [Build Dec] point_dim=3, context_dim=1
2026-03-19 14:59:21.323 | INFO     | models.latent_points_ada:__init__:38 - [Build Unet] extra_feature_channels=1, input_dim=3


ninja: no work to do.
Loading extension module _pvcnn_backend...


2026-03-19 14:59:21.435 | INFO     | models.vae_adain:__init__:50 - [Build Model] style_encoder: models.shapelatent_modules.PointNetPlusEncoder, encoder: models.latent_points_ada.PointTransPVC, decoder: models.latent_points_ada.LatentPointDecPVC


Model(
  (style_encoder): PointNetPlusEncoder(
    (mlp): Linear(in_features=64, out_features=256, bias=True)
    (layers): ModuleList(
      (0): Sequential(
        (0): PVConv(
          (voxelization): Voxelization(resolution=32, normalized eps = 0)
          (voxel_layers): Sequential(
            (0): Conv3d(3, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
            (1): GroupNorm(8, 32, eps=1e-05, affine=True)
            (2): Swish()
            (3): Dropout(p=0.1, inplace=False)
            (4): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
            (5): GroupNorm(8, 32, eps=1e-05, affine=True)
            (6): SE(32, 32)
          )
          (point_features): SharedMLP(
            (layers): Sequential(
              (0): Conv1d(3, 32, kernel_size=(1,), stride=(1,))
              (1): GroupNorm(8, 32, eps=1e-05, affine=True)
              (2): Swish()
            )
          )
        )
        (1): PVConv(
          (voxel

In [20]:
# Load real cloud
x1 = np.load(shape1)
x2 = np.load(shape2)

print(x1.ndim, x2.ndim)
print(x1.shape, x2.shape)

if x1.ndim == 2 and x2.ndim == 2: # Matriz
    x1 = x1[None, ...]
    x2 = x2[None, ...]
    print(x1.shape, x2.shape)

x1_tensor = torch.tensor(x1, dtype=torch.float32).to(device)
x2_tensor = torch.tensor(x2, dtype=torch.float32).to(device)

x1_latent_list = vae.get_latents(x1_tensor)
x2_latent_list = vae.get_latents(x2_tensor)

#print("1: ", x1_latent_list[0][0])

# Vector distance 
dist = torch.norm(x1_latent_list[0][0] - x2_latent_list[0][0], dim=-1) # Euclidian distance
print("Distance:", dist.mean()) # Embeddings are far

# Measures if these embeddings point in the same direction
cosine_similarity = torch.nn.functional.cosine_similarity(x1_latent_list[0][0], x2_latent_list[0][0], dim=-1)
print("Cosine similarity", cosine_similarity.mean()) # They are closely similar

# Average difference per dimension
mean_square_error = torch.mean((x1_latent_list[0][0] - x2_latent_list[0][0])**2)
print("Mean square error", mean_square_error) # Moderate difference

2 2
(15000, 3) (15000, 3)
(1, 15000, 3) (1, 15000, 3)
Distance: tensor(6.3260, device='cuda:0')
Cosine similarity tensor(0.7406, device='cuda:0')
Mean square error tensor(0.3126, device='cuda:0')


In [ ]:
""" # Obtain h_0
with torch.no_grad():
    h0 = vae.get_latent_points(x_tensor)

np.save(ouput_path, h0.squeeze(0).cpu().numpy())
print("done") """